# MuseumSCAT: fine-tune doctr's Danish-vocab recognizer

Before running: attach the private dataset `museumscat-danish-recognizer-crops` as input
(Notebook Settings → Input → search for it → Add), and under Settings enable
**Accelerator = GPU T4 x2 (or P100)** and **Internet = On**.

**Two-stage recipe** (see `docs/superpowers/plans/2026-07-24-danish-recognizer-finetune.md`,
"Fine-tune result" section): the first frozen-backbone attempt (20 epochs, 240 gradient
steps) left the reinitialized vocab-danish classifier head undertrained — every date
prediction came back MISSING and recognized text was garbled noise. Stage 1 below gives
the head far more steps to actually learn; Stage 2 unfreezes the whole network at a
10x lower LR so the conv backbone can adapt slightly without wrecking its pretrained
features. Both stages use early stopping.

In [ ]:
!pip install -q "python-doctr[torch]==1.0.1" matplotlib
!mkdir -p doctr_recognition
!curl -s -o doctr_recognition/train.py "https://raw.githubusercontent.com/mindee/doctr/v1.0.1/references/recognition/train.py"
!curl -s -o doctr_recognition/utils.py "https://raw.githubusercontent.com/mindee/doctr/v1.0.1/references/recognition/utils.py"

In [ ]:
import os

INPUT_DIR = "/kaggle/input/datasets/matiaspoullain/museumscat-danish-recognizer-crops"
TRAIN_PATH = os.path.join(INPUT_DIR, "train")
VAL_PATH = os.path.join(INPUT_DIR, "val")
assert os.path.isfile(os.path.join(TRAIN_PATH, "labels.json"))
assert os.path.isfile(os.path.join(VAL_PATH, "labels.json"))

In [ ]:
!cd doctr_recognition && python train.py crnn_vgg16_bn \
    --vocab danish --pretrained --freeze-backbone \
    --train_path {TRAIN_PATH} --val_path {VAL_PATH} \
    --epochs 150 --batch_size 32 --lr 0.001 \
    --early-stop --early-stop-epochs 15 --early-stop-delta 0.001 \
    --output_dir /kaggle/working --name danish_crnn_vgg16_bn_stage1

In [ ]:
import os

assert os.path.isfile("/kaggle/working/danish_crnn_vgg16_bn_stage1.pt")
print("stage 1 checkpoint ready: /kaggle/working/danish_crnn_vgg16_bn_stage1.pt")

In [ ]:
!cd doctr_recognition && python train.py crnn_vgg16_bn \
    --vocab danish --resume /kaggle/working/danish_crnn_vgg16_bn_stage1.pt \
    --train_path {TRAIN_PATH} --val_path {VAL_PATH} \
    --epochs 50 --batch_size 32 --lr 0.0001 \
    --early-stop --early-stop-epochs 10 --early-stop-delta 0.001 \
    --output_dir /kaggle/working --name danish_crnn_vgg16_bn_stage2

In [ ]:
import os

assert os.path.isfile("/kaggle/working/danish_crnn_vgg16_bn_stage2.pt")
print("final checkpoint ready to download: /kaggle/working/danish_crnn_vgg16_bn_stage2.pt")